# Red-Eye Disease Triage: Classifier + Citation-Constrained LLM Guidance

End-to-end driver notebook for the analyses reported in *Integrating Deep Learning and Large Language Models for Community-Based Triage of Red Eye Diseases* (submitted to *Translational Vision Science & Technology*).

All logic lives in `src/redeye_triage/`; this notebook only sets paths and calls it, so the notebook and the command-line scripts in `scripts/` produce identical results.

| Step | What it does | Manuscript items |
|---|---|---|
| 1 | Stratified 85/15 split, 5-fold CV, bootstrap CIs, per-class metrics, operating points, calibration, held-out refit | Tables 1-3, Supplementary Tables S1-S2, Supplementary Figures S1-S2 |
| 2 | Exact/near-duplicate audit and train/test leakage check | Methods (curation), Limitations |
| 3 | Grad-CAM++, Integrated Gradients, nearest-neighbour retrieval | Figure 2 |
| 4 | Bootstrap model-comparison figures | - |
| 5 | Citation-constrained Gemini guidance + grounding audit | Figure 3, Supplementary Table S3 |

> **Research use only.** This software is not a medical device and has not been validated for clinical decision-making.

## 0. Setup
Runs in Google Colab (GPU runtime recommended) or locally. In Colab, the repository is cloned and images are read from Google Drive.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/<your-github-username>/red-eye-triage.git"   # <-- update after creating the repository

if IN_COLAB:
    if not os.path.exists("red-eye-triage"):
        !git clone {REPO_URL}
    %cd red-eye-triage
    !pip install -q -r requirements.txt

In [ ]:
# ---- Paths: set BEFORE importing redeye_triage ----
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["REDEYE_DATA_DIR"]   = "/content/drive/MyDrive/ML External Eye Prediction/data_2"
    os.environ["REDEYE_OUTPUT_DIR"] = "/content/drive/MyDrive/ML External Eye Prediction/publication_outputs"
else:
    os.environ.setdefault("REDEYE_DATA_DIR", os.path.abspath("../data/images"))
    os.environ.setdefault("REDEYE_OUTPUT_DIR", os.path.abspath("../outputs"))

# os.environ["REDEYE_FAST_MODE"] = "1"   # uncomment for a quick debug run (NOT for reported results)

repo_src = os.path.abspath("src" if os.path.isdir("src") else "../src")
sys.path.insert(0, repo_src)

from redeye_triage import config as C
print("Data:", C.DATASET_PATH)
print("Outputs:", C.OUTPUT_DIR)
print("Device:", C.get_device(), "| FAST_MODE:", C.FAST_MODE)

## 1. Training and evaluation
If `outputs/checkpoints/split_indices.npz` already exists it is reused, so the reported split is preserved. Models with saved results are skipped unless `force=True`.

In [ ]:
from redeye_triage.pipeline import prepare_data, run_one_model, assemble_tables

data, labels, manifest, idx_train, idx_test = prepare_data()
for m in C.MODELS_TO_RUN:
    run_one_model(m, data, labels, idx_train, idx_test, force=False)
assemble_tables()

In [ ]:
import pandas as pd
pd.read_csv(os.path.join(C.OUTPUT_DIR, "table1_overall_metrics_cv.csv"))

## 2. Duplicate and leakage audit

In [ ]:
from redeye_triage.duplicates import run_duplicate_audit
hashes, exact_dups, near_dups = run_duplicate_audit()
near_dups.head(20)

## 3. Explainability (Figure 2)

In [ ]:
import numpy as np
from redeye_triage.data import EyeDiseaseDataset, get_transforms
from redeye_triage.models import load_checkpoint
from redeye_triage.explainability import (build_feature_extractor, build_reference_index,
                                          explain_case, gradcam_class_grid)

MODEL_NAME = "efficientnet"
_, eval_tf = get_transforms()
test_ds = EyeDiseaseDataset(data[idx_test], labels[idx_test], eval_tf)
model = load_checkpoint(MODEL_NAME, os.path.join(C.CKPT_DIR, f"{MODEL_NAME}_heldout_refit.pth"))

feat = build_feature_extractor(model, MODEL_NAME)
index, _, kept_imgs, kept_labels = build_reference_index(feat, data[idx_train], labels[idx_train], eval_tf)  # TRAIN only

os.makedirs(C.EXPLAIN_DIR, exist_ok=True)
gradcam_class_grid(model, MODEL_NAME, test_ds, save_path=os.path.join(C.EXPLAIN_DIR, f"fig_gradcam_grid_{MODEL_NAME}.png"))
for ci in sorted({int(i) for i in np.linspace(0, len(test_ds) - 1, 3)}):
    img_t, lab = test_ds[ci]
    info = explain_case(model, MODEL_NAME, feat, index, kept_imgs, kept_labels, img_t, true_label=lab,
                        save_path=os.path.join(C.EXPLAIN_DIR, f"fig_explanation_{MODEL_NAME}_case{ci}.png"))
    print(ci, manifest.iloc[int(idx_test[ci])]["path"], info["ig_convergence_delta"])

## 4. Model-comparison figures

In [ ]:
from redeye_triage.plots import plot_metric_comparison, plot_bootstrap_distributions
boot = os.path.join(C.OUTPUT_DIR, "metrics_bootstrap_raw.json")
os.makedirs(C.COMPARISON_DIR, exist_ok=True)
plot_metric_comparison(boot, save_path=os.path.join(C.COMPARISON_DIR, "fig_model_comparison_ci.png"))
for metric in ("auroc_ovr", "auprc_macro"):
    plot_bootstrap_distributions(boot, metric, save_path=os.path.join(C.COMPARISON_DIR, f"fig_{metric}_bootstrap.png"))

## 5. Citation-constrained LLM guidance (Figure 3)
Store your key as a Colab secret named `GEMINI_API_KEY` (or export it locally). **Never paste API keys into the notebook.** Every generation is saved with its prompt, output, audit result, model name and UTC timestamp in `outputs/llm_outputs/`.

In [ ]:
from redeye_triage import llm_grounding as L

if IN_COLAB:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

kb = L.load_knowledge_base()
system, user, items = L.build_grounded_prompt("Inflammatory", kb, 0.90)
print(user[:1500])

In [ ]:
import torch, torch.nn.functional as F

client = L.get_gemini_client()

# (A) one demonstration output per class
for cls in C.CLASSES:
    rec = L.generate_grounded_instruction(client, cls, kb, class_probability=0.90)
    L.save_generation_record(rec, name=f"demo_{cls}")
    print("=" * 70, f"\n{cls}\n" + "=" * 70)
    print(rec["output_text"])
    print("[audit]", {k: rec["grounding_audit"][k] for k in ("grounding_pass", "n_fabricated_refs", "n_uncited_clinical_lines")})

# (B) held-out image -> classifier prediction -> grounded guidance
CASE = 0
img_t, true_label = test_ds[CASE]
with torch.no_grad():
    probs = F.softmax(model(img_t.unsqueeze(0).to(C.get_device())), dim=1).cpu().numpy().ravel()
pred = int(probs.argmax())
rec = L.generate_grounded_instruction(client, C.CLASSES[pred], kb, class_probability=float(probs[pred]))
rec.update({"test_case": CASE, "true_class": C.CLASSES[int(true_label)],
            "class_probabilities": dict(zip(C.CLASSES, probs.tolist()))})
print(L.save_generation_record(rec, name=f"test_case_{CASE}"))
print(rec["output_text"])

## 6. Record the software environment
Save exact package versions next to the results so they can be reported and pinned.

In [ ]:
!pip freeze > "{C.OUTPUT_DIR}/environment_pip_freeze.txt"
import platform, torch, torchvision
print(platform.python_version(), torch.__version__, torchvision.__version__,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")